In [ ]:
import os, time
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, APIStatusError

load_dotenv(
    r"C:\Users\kuro\MAST_RADP\.env",
    override=True
)  # reads settings + your API key from the .env file

PROVIDERS = {
    "groq":   {"base_url": "https://api.groq.com/openai/v1", "key_env": "GROQ_API_KEY"},
    "openai": {"base_url": None,                             "key_env": "OPENAI_API_KEY"},
    "ollama": {"base_url": "http://localhost:11434/v1",      "key_env": None},
}
PROVIDER          = os.getenv("LLM_PROVIDER", "groq").lower()
MODEL             = os.getenv("LLM_MODEL", "openai/gpt-oss-120b")
MAX_PROMPT_TOKENS = int(os.getenv("MAX_PROMPT_TOKENS", "6500"))
USE_EXAMPLES      = os.getenv("USE_EXAMPLES", "false").lower() == "true"

cfg = PROVIDERS[PROVIDER]
api_key = os.getenv(cfg["key_env"]) if cfg["key_env"] else "ollama"
if not api_key or api_key.startswith("paste-"):
    raise ValueError(f"Put your {cfg['key_env']} in the .env file (copy .env.example to .env first).")
client = OpenAI(base_url=cfg["base_url"], api_key=api_key)
print(f"Using {PROVIDER} / {MODEL}  (prompt budget ~{MAX_PROMPT_TOKENS} tokens, examples={'on' if USE_EXAMPLES else 'off'})")

def chat_completion_request_openai(prompt, retries=6):
    for attempt in range(retries):
        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=1.0)
            return r.choices[0].message.content if r.choices else None
        except RateLimitError as e:  # 429: per-minute limit hit -> wait and retry
            wait = float(e.response.headers.get("retry-after", 30)) + 2
            print(f"  rate limited, waiting {wait:.0f}s ...")
            time.sleep(wait)
        except APIStatusError as e:
            if e.status_code == 413:
                raise RuntimeError("Prompt is bigger than your per-minute token limit. "
                                   "Lower MAX_PROMPT_TOKENS in .env.") from e
            raise
    raise RuntimeError("Still rate limited after several retries (daily limit reached?).")

definitions = open("taxonomy_definitions_examples/definitions.txt", "r").read()

def openai_evaluator(trace, definitions=definitions, examples=''):
    prompt = (
    "Below I will provide a multiagent system trace. provide me an analysis of the failure modes and inefficiencies as I will say below. \n"
    "In the traces, analyze the system behaviour."
    "There are several failure modes in multiagent systems I identified. I will provide them below. Tell me if you encounter any of them, as a binary yes or no. \n"
    "Also, give me a one sentence (be brief) summary of the problems with the inefficiencies or failure modes in the trace. Only mark a failure mode if you can provide an example of it in the trace, and specify that in your summary at the end"
    "Also tell me whether the task is successfully completed or not, as a binary yes or no."
    "At the very end, I provide you with the definitions of the failure modes and inefficiencies. After the definitions, I will provide you with examples of the failure modes and inefficiencies for you to understand them better."
    "Tell me if you encounter any of them between the @@ symbols as I will say below, as a binary yes or no."
    "Here are the things you should answer. Start after the @@ sign and end before the next @@ sign (do not include the @@ symbols in your answer):"
    "*** begin of things you should answer *** @@"
    "A. Freeform text summary of the problems with the inefficiencies or failure modes in the trace: <summary>"
    "B. Whether the task is successfully completed or not: <yes or no>"
    "C. Whether you encounter any of the failure modes or inefficiencies:"
    "1.1 Disobey Task Specification: <yes or no>"
    "1.2 Disobey Role Specification: <yes or no>"
    "1.3 Step Repetition: <yes or no>"
    "1.4 Loss of Conversation History: <yes or no>"
    "1.5 Unaware of Termination Conditions: <yes or no>"
    "2.1 Conversation Reset: <yes or no>"
    "2.2 Fail to Ask for Clarification: <yes or no>"
    "2.3 Task Derailment: <yes or no>"
    "2.4 Information Withholding: <yes or no>"
    "2.5 Ignored Other Agent's Input: <yes or no>"
    "2.6 Action-Reasoning Mismatch: <yes or no>"
    "3.1 Premature Termination: <yes or no>"
    "3.2 No or Incorrect Verification: <yes or no>"
    "3.3 Weak Verification: <yes or no>"
    "@@*** end of your answer ***"
    "An example answer is: \n"
    "A. The task is not completed due to disobeying role specification as agents went rogue and started to chat with each other instead of completing the task. Agents derailed and verifier is not strong enough to detect it.\n"
    "B. no \n"
    "C. \n"
    "1.1 no \n"
    "1.2 no \n"
    "1.3 no \n"
    "1.4 no \n"
    "1.5 no \n"
    "1.6 yes \n"
    "2.1 no \n"
    "2.2 no \n"
    "2.3 yes \n"
    "2.4 no \n"
    "2.5 no \n"
    "2.6 yes \n"
    "2.7 no \n"
    "3.1 no \n"
    "3.2 yes \n"
    "3.3 no \n"   
    "Here is the trace: \n"
    f"{trace}"
    "Also, here are the explanations (definitions) of the failure modes and inefficiencies: \n"
    f"{definitions} \n"
    "Here are some examples of the failure modes and inefficiencies: \n"
    f"{examples}"
)
    return chat_completion_request_openai(prompt)

Using groq / openai/gpt-oss-120b  (prompt budget ~2000 tokens, examples=off)


In [2]:
# The examples file is ~17K tokens: too big for Groq's free tier, so it is off by default.
examples = open("taxonomy_definitions_examples/examples.txt", "r", encoding="utf-8").read() if USE_EXAMPLES else ""

## LLM As a Judge Experiments

In [3]:
import pandas as pd

In [4]:
import glob

TRACE_GLOB = os.getenv("TRACE_GLOB", "traces/mmlu/metagpt_mmlu/*.txt")
MAX_TRACES = int(os.getenv("MAX_TRACES", "30"))

trace_files = sorted(glob.glob(TRACE_GLOB))[:MAX_TRACES]
full_trace_list = [open(f, "r", encoding="utf-8", errors="ignore").read() for f in trace_files]
print(f"Loaded {len(full_trace_list)} traces from {TRACE_GLOB}")

Loaded 3 traces from traces/mmlu/metagpt_mmlu/*.txt


In [5]:
openai_results = []

In [6]:
import pickle
import os

def approx_tokens(text):
    return int(len(text) / 3.5)  # slightly conservative (real tokenizers avg ~4 chars/token)

def fit_trace(trace):
    """Trim a trace so the whole prompt fits the token budget.
    Keeps the beginning (the task) and the end (how it finished), since
    many failure modes (termination, verification) only show up at the end."""
    overhead = approx_tokens(definitions + examples) + 1000   # instructions + answer room
    budget_chars = int(max(300, MAX_PROMPT_TOKENS - overhead) * 3.5)
    if len(trace) <= budget_chars:
        return trace
    half = budget_chars // 2
    return trace[:half] + "\n\n[... middle of trace trimmed to fit token limit ...]\n\n" + trace[-half:]

dirname = 'saved_results'
os.makedirs(dirname, exist_ok=True)

for i in range(len(full_trace_list)):
    
    trace = fit_trace(full_trace_list[i])


    try:
        openai_evaluation = openai_evaluator(trace, examples=examples)
        openai_results.append(openai_evaluation)
        
        # Save the current results after each evaluation
        with open(f'{dirname}/results_checkpoint.pkl', 'wb') as f:
            pickle.dump(openai_results, f)
            
        # Optional: Save a backup copy every 10 evaluations
        if (i + 1) % 10 == 0:
            with open(f'{dirname}/results_backup_{i+1}.pkl', 'wb') as f:
                pickle.dump(openai_results, f)
                
        print(f"Completed and saved evaluation {i+1}/{len(full_trace_list)}")
    except Exception as e:
        print(f"Error on evaluation {i+1}: {str(e)}")
        # Save results even if there's an error
        with open(f'{dirname}/results_checkpoint.pkl', 'wb') as f:
            pickle.dump(openai_results, f)

Completed and saved evaluation 1/3
Completed and saved evaluation 2/3
Completed and saved evaluation 3/3


In [12]:
o1_results = openai_results

In [13]:
print(
    f"Successful evaluations: "
    f"{len(openai_results)}/{len(full_trace_list)}"
)

Successful evaluations: 3/3


In [14]:
def parse_responses(responses):
    """
    Parse the LLM responses to extract yes/no answers for each failure mode.
    
    Args:
        responses: List of LLM responses evaluating traces
        
    Returns:
        Dictionary mapping failure mode codes to lists of binary values (0 for no, 1 for yes)
    """
    import re
    
    # Initialize dictionary with empty lists for each failure mode
    failure_modes = {
        '1.1': [], '1.2': [], '1.3': [], '1.4': [], '1.5': [],
        '2.1': [], '2.2': [], '2.3': [], '2.4': [], '2.5': [], '2.6': [],
        '3.1': [], '3.2': [], '3.3': []
    }
    
    for i, response in enumerate(responses):
        try:
            # Clean up the response - remove @@ markers if present
            cleaned_response = response.strip()
            if cleaned_response.startswith('@@'):
                cleaned_response = cleaned_response[2:]
            if cleaned_response.endswith('@@'):
                cleaned_response = cleaned_response[:-2]
            
            # Process each failure mode
            for mode in failure_modes.keys():
                # Various patterns to match different response formats
                patterns = [
                    # Format with C. prefix and colon
                    rf"C\..*?{mode}.*?(yes|no)",
                    # Format with just C prefix without dot
                    rf"C{mode}\s+(yes|no)",
                    # Format with mode directly (with or without spaces)
                    rf"{mode}\s*[:]\s*(yes|no)",
                    rf"{mode}\s+(yes|no)",
                    # Format with newlines
                    rf"{mode}\s*\n\s*(yes|no)",
                    # Format with C prefix and newlines
                    rf"C\.{mode}\s*\n\s*(yes|no)"
                ]
                
                found = False
                for pattern in patterns:
                    matches = re.findall(pattern, cleaned_response, re.IGNORECASE | re.DOTALL)
                    if matches:
                        # Use the first match
                        value = 1 if matches[0].lower() == 'yes' else 0
                        failure_modes[mode].append(value)
                        found = True
                        break
                
                if not found:
                    # If we still can't find a match, try a more general approach
                    # Look for the mode number followed by any text and then yes/no
                    general_pattern = rf"(?:C\.)?{mode}.*?(yes|no)"
                    match = re.search(general_pattern, cleaned_response, re.IGNORECASE | re.DOTALL)
                    
                    if match:
                        value = 1 if match.group(1).lower() == 'yes' else 0
                        failure_modes[mode].append(value)
                    else:
                        # If all attempts fail, default to 'no'
                        print(f"Warning: Could not find mode {mode} in response {i}")
                        failure_modes[mode].append(0)
                    
        except Exception as e:
            print(f"Error processing response {i}: {e}")
            # If there's an error, default to 'no' for all modes for this response
            for mode in failure_modes:
                if len(failure_modes[mode]) <= i:  # Only append if we haven't already
                    failure_modes[mode].append(0)
    
    # Ensure all lists have the same length
    max_length = max(len(values) for values in failure_modes.values())
    for mode in failure_modes:
        if len(failure_modes[mode]) < max_length:
            failure_modes[mode].extend([0] * (max_length - len(failure_modes[mode])))
    
    return failure_modes


In [15]:
# Import regex for pattern matching
import re

# Process the OpenAI evaluation results
failure_mode_results_o1 = parse_responses(o1_results)

# Print the first few entries of each failure mode to verify
for mode, values in failure_mode_results_o1.items():
    print(f"{mode}: {values[:5]} (total yes: {sum(values)}/{len(values)}, {round(sum(values)/len(values)*100, 2)}%)")

1.1: [1, 1, 1] (total yes: 3/3, 100.0%)
1.2: [1, 0, 1] (total yes: 2/3, 66.67%)
1.3: [0, 0, 0] (total yes: 0/3, 0.0%)
1.4: [0, 0, 0] (total yes: 0/3, 0.0%)
1.5: [0, 0, 1] (total yes: 1/3, 33.33%)
2.1: [0, 0, 0] (total yes: 0/3, 0.0%)
2.2: [1, 1, 1] (total yes: 3/3, 100.0%)
2.3: [1, 1, 1] (total yes: 3/3, 100.0%)
2.4: [0, 0, 0] (total yes: 0/3, 0.0%)
2.5: [0, 0, 0] (total yes: 0/3, 0.0%)
2.6: [1, 0, 1] (total yes: 2/3, 66.67%)
3.1: [0, 0, 0] (total yes: 0/3, 0.0%)
3.2: [0, 0, 1] (total yes: 1/3, 33.33%)
3.3: [0, 0, 0] (total yes: 0/3, 0.0%)


In [16]:
# Define the sizes of each source
source_sizes = [len(full_trace_list)]  # one group per trace folder you loaded; e.g. [30, 30] for two folders

# Calculate the starting index for each source
source_indices = [0] #[0, 28, 58, 88, 119]
for size in source_sizes[:-1]:
    source_indices.append(source_indices[-1] + size)

# print(source_indices)
# Dictionary to store average scores for each failure mode across sources
average_scores_by_source = {}

# Calculate average scores for each failure mode across the 5 sources
for mode, values in failure_mode_results_o1.items():
    source_averages = []
    # print(len(values))

    # Calculate average for each source
    for i in range(len(source_sizes)):
        start_idx = source_indices[i]
        end_idx = start_idx + source_sizes[i]
        if i == 4:
            end_idx = len(values)
        source_values = values[start_idx:end_idx]
        
                # Check if source_values is not empty to avoid division by zero
        if len(source_values) > 0:
            avg_score = sum(source_values) / len(source_values)
        else:
            print(f"Warning: Source {i} has no values for mode {mode}")
            avg_score = 0
            
        source_averages.append(avg_score)
    
    average_scores_by_source[mode] = source_averages
    
    # Print the average scores for each source
    print(f"{mode}: {[round(score * 100, 2) for score in source_averages]}%")

# Print a sample of the dictionary structure
print("\nSample of average_scores_by_source dictionary:")
sample_key = list(average_scores_by_source.keys())[0]
print(f"{sample_key}: {average_scores_by_source[sample_key]}")


1.1: [100.0]%
1.2: [66.67]%
1.3: [0.0]%
1.4: [0.0]%
1.5: [33.33]%
2.1: [0.0]%
2.2: [100.0]%
2.3: [100.0]%
2.4: [0.0]%
2.5: [0.0]%
2.6: [66.67]%
3.1: [0.0]%
3.2: [33.33]%
3.3: [0.0]%

Sample of average_scores_by_source dictionary:
1.1: [1.0]


In [17]:
print(f"Successful evaluations: {len(openai_results)}/{len(full_trace_list)}")

Successful evaluations: 3/3


In [18]:
print(f"Successful responses: {len(openai_results)}/{len(full_trace_list)}")

for i, response in enumerate(openai_results, start=1):
    print(f"\n{'=' * 20} TRACE {i} {'=' * 20}")
    print(response)

Successful responses: 3/3

==================== TRACE 1 ====================
@@
A. The agents deviated from the required concise answer format, added unnecessary explanatory content, did not request clarification for the truncated question, and thus failed to adhere to the task and role specifications.  
B. no  
C.  
1.1 Disobey Task Specification: yes  
1.2 Disobey Role Specification: yes  
1.3 Step Repetition: no  
1.4 Loss of Conversation History: no  
1.5 Unaware of Termination Conditions: no  
2.1 Conversation Reset: no  
2.2 Fail to Ask for Clarification: yes  
2.3 Task Derailment: yes  
2.4 Information Withholding: no  
2.5 Ignored Other Agent's Input: no  
2.6 Action-Reasoning Mismatch: yes  
3.1 Premature Termination: no  
3.2 No or Incorrect Verification: no  
3.3 Weak Verification: no  
@@

==================== TRACE 2 ====================
@@
A. The agents ignored the required answer format, did not ask for clarification on the incomplete question, and drifted into unrelated

In [19]:
for path in trace_files:
    print(path)

traces/mmlu/metagpt_mmlu\mmlu_0.txt
traces/mmlu/metagpt_mmlu\mmlu_1.txt
traces/mmlu/metagpt_mmlu\mmlu_10.txt


In [20]:
import json
from pathlib import Path

output_path = Path("saved_results/judge_pilot_review.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

records = [
    {
        "trace_file": path,
        "original_characters": len(trace),
        "sent_characters": len(fit_trace(trace)),
        "trace_was_trimmed": fit_trace(trace) != trace,
        "judge_response": response,
    }
    for path, trace, response in zip(
        trace_files, full_trace_list, openai_results
    )
]

output_path.write_text(
    json.dumps(records, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Saved to: {output_path.resolve()}")

Saved to: C:\Users\kuro\MAST_RADP\saved_results\judge_pilot_review.json


In [21]:
print("Approximate tokens; not exact model-tokenizer counts")
print("Definitions:", approx_tokens(definitions))
print("Examples:", approx_tokens(examples))

for path, trace in zip(trace_files, full_trace_list):
    estimate = approx_tokens(definitions + examples + trace) + 1000
    print(f"{path}: estimated budget needed ≈ {estimate}")

Approximate tokens; not exact model-tokenizer counts
Definitions: 3760
Examples: 0
traces/mmlu/metagpt_mmlu\mmlu_0.txt: estimated budget needed ≈ 6567
traces/mmlu/metagpt_mmlu\mmlu_1.txt: estimated budget needed ≈ 6181
traces/mmlu/metagpt_mmlu\mmlu_10.txt: estimated budget needed ≈ 6338
